# NOORIX-AI Urdu / English Speech-to-Text — Whisper Fine-tuning
**Prepared by Muhammad Farhan**

Fine-tune a multilingual Whisper checkpoint for one language per run. This is an ASR notebook (speech → text), not text-to-speech.

## Kaggle notebook title
`NOORIX-ASR-Whisper-Urdu-English-Muhammad-Farhan`

## Dataset to add
- Urdu: Search Kaggle for `Mozilla Common Voice Urdu` or `Common Voice Corpus Urdu audio`. An alternative official dataset source is Mozilla Data Collective's Common Voice Urdu release; it contains audio clips and TSV transcriptions. Check its current access and terms.
- English: Search Kaggle for `Common Voice English`.
- Your own recordings: use a CSV with `audio_path,text` columns. Audio must be clear, consented speech, and transcription must match exactly.

Choose `LANGUAGE="urdu"` or `LANGUAGE="english"` in the config cell. Do not mix languages in one run without a deliberate multilingual data strategy. Whisper is large; use a Kaggle GPU and begin with a small subset.


In [ ]:
!pip -q install -U transformers datasets[audio] evaluate jiwer accelerate soundfile
import os, glob, json, shutil
from pathlib import Path
import pandas as pd, numpy as np, torch
from datasets import Dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Attach a Kaggle dataset containing audio and a transcription CSV.
for p in glob.glob("/kaggle/input/*"): print(p)
CSV_PATH="" # path to CSV/TSV containing audio_path,text
AUDIO_COL="audio_path"; TEXT_COL="text"
LANGUAGE="urdu" # set to "english" for English run
MODEL_NAME="openai/whisper-small"
if not CSV_PATH: raise ValueError("Set CSV_PATH to your labelled audio manifest.")
sep="\t" if CSV_PATH.lower().endswith(".tsv") else ","
df=pd.read_csv(CSV_PATH,sep=sep)
df=df[[AUDIO_COL,TEXT_COL]].dropna()
df.columns=["audio_path","text"]
def resolve(p):
    p=Path(str(p))
    if p.is_file(): return str(p)
    for root in Path("/kaggle/input").glob("*"):
        q=root/p
        if q.is_file(): return str(q)
        hits=list(root.rglob(p.name))
        if hits: return str(hits[0])
    return None
df["audio_path"]=df.audio_path.apply(resolve)
df=df.dropna().reset_index(drop=True)
print("Usable clips:",len(df)); display(df.head())
if len(df)<100: raise ValueError("Need more labelled audio rows before fine-tuning.")


In [ ]:
# Limit initially for faster first test; increase after pipeline works.
MAX_SAMPLES=5000
df=df.sample(min(len(df),MAX_SAMPLES),random_state=42).reset_index(drop=True)
ds=Dataset.from_pandas(df)
ds=ds.rename_column("audio_path","audio")
ds=ds.cast_column("audio",Audio(sampling_rate=16000))
splits=ds.train_test_split(test_size=0.1,seed=42)
processor=WhisperProcessor.from_pretrained(MODEL_NAME,language=LANGUAGE,task="transcribe")
model=WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language=LANGUAGE
model.generation_config.task="transcribe"
model.generation_config.forced_decoder_ids=None


In [ ]:
def prepare(batch):
    audio=batch["audio"]
    batch["input_features"]=processor.feature_extractor(audio["array"],sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"]=processor.tokenizer(batch["text"]).input_ids
    return batch
train_ds=splits["train"].map(prepare,remove_columns=splits["train"].column_names)
val_ds=splits["test"].map(prepare,remove_columns=splits["test"].column_names)
from dataclasses import dataclass
@dataclass
class DataCollator:
    processor: object
    def __call__(self,features):
        inputs=[{"input_features":f["input_features"]} for f in features]
        batch=self.processor.feature_extractor.pad(inputs,return_tensors="pt")
        labels=[{"input_ids":f["labels"]} for f in features]
        lab=self.processor.tokenizer.pad(labels,return_tensors="pt")
        batch["labels"]=lab["input_ids"].masked_fill(lab.attention_mask.ne(1),-100)
        return batch
collator=DataCollator(processor)


In [ ]:
cer=evaluate.load("cer"); wer=evaluate.load("wer")
def metrics_fn(pred):
    pred_ids=pred.predictions[0] if isinstance(pred.predictions,tuple) else pred.predictions
    label_ids=pred.label_ids.copy(); label_ids[label_ids==-100]=processor.tokenizer.pad_token_id
    p=processor.tokenizer.batch_decode(pred_ids,skip_special_tokens=True)
    r=processor.tokenizer.batch_decode(label_ids,skip_special_tokens=True)
    return {"cer":cer.compute(predictions=p,references=r),"wer":wer.compute(predictions=p,references=r)}
args=Seq2SeqTrainingArguments(output_dir="/kaggle/working/noorix_asr_ckpt",per_device_train_batch_size=4,
    per_device_eval_batch_size=4,gradient_accumulation_steps=2,learning_rate=1e-5,num_train_epochs=2,
    fp16=torch.cuda.is_available(),eval_strategy="epoch",save_strategy="epoch",predict_with_generate=True,
    generation_max_length=128,logging_steps=25,save_total_limit=2,load_best_model_at_end=True,
    metric_for_best_model="cer",greater_is_better=False,report_to="none")
trainer=Seq2SeqTrainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,
    data_collator=collator,compute_metrics=metrics_fn,processing_class=processor)
trainer.train()


In [ ]:
metrics=trainer.evaluate(); print(metrics)
out=Path("/kaggle/working/NOORIX_ASR_Whisper_"+LANGUAGE+"_Muhammad_Farhan"); out.mkdir(exist_ok=True)
model.save_pretrained(out/"model"); processor.save_pretrained(out/"processor")
(out/"metadata.json").write_text(json.dumps({"project":"NOORIX-AI","prepared_by":"Muhammad Farhan",
"task":"ASR","language":LANGUAGE,"base_model":MODEL_NAME,"validation_metrics":{k:float(v) for k,v in metrics.items() if isinstance(v,(int,float))},
"note":"Validate on a separate real-world test set; local CPU latency and memory must be measured."},indent=2),encoding="utf-8")
z=shutil.make_archive("/kaggle/working/NOORIX_ASR_Whisper_"+LANGUAGE+"_Muhammad_Farhan","zip",out)
print("ZIP:",z)
